# 1.10 Wrap-up of Part 1

In Part 1 of this module, we:

- Learned what RAG is and why it matters: retrieve documents, build a prompt, let the LLM generate a grounded answer
- Built a search engine over a real FAQ dataset using minsearch
- Created a prompt template that combines the user's question with search results
- Wired search + prompt + LLM into a working RAG pipeline
- Split ingestion and query into separate processes with sqlitesearch

You now have a working RAG system and a clear mental model for how each piece fits together. From here, the work is making each piece better.

## Two directions forward

Part 2 of this module: Agents. Our pipeline runs search once with the exact user query, unchanged. If that search returns garbage, the LLM has no way to recover. An agent puts the LLM in charge instead. It decides what to search for, how many searches to run, and when to stop.

An agent also handles questions in another language. It translates the query before searching, then translates the answer back afterward.

Module 2: Vector Search. Keyword matches are exact. Vector search matches by semantic meaning instead, which helps when the user phrases things differently from the FAQ.

## Elasticsearch

Elasticsearch is the industry standard for text search.

It supports:

- Full-text search with BM25
- Filtering, aggregations, and faceting
- Vector search (dense and sparse)
- Distributed scaling
- Real-time indexing

It's heavier than sqlitesearch but handles production workloads at scale. If you're building a real RAG system, Elasticsearch (or OpenSearch) is a common choice for the search backend.

For an Elasticsearch tutorial, see the supplementary materials for Module 1.

## Fine-tuning vs RAG

Instead of retrieving documents at query time, you could fine-tune the LLM on your data.

Fine-tuning means taking a model's weights and adjusting them for your specific use case.

This works, but it has downsides:

- It requires special hardware (GPUs) and tools we don't cover in this course
- It's difficult to update when new data arrives - you don't want to retrain the model every time a new FAQ entry is added
- The LLM already has internal knowledge, but RAG gives it access to information it wasn't trained on

RAG is more flexible, cheaper, and works with any LLM. In practice, fine-tuning is rarely needed. I've never personally hit a case that required it. When I analyzed around 2,500 job descriptions for my AI engineering field guide, few asked for it. So focus on RAG first, and reach for fine-tuning only when you genuinely need it.

## Moving forward

ry these next steps:

- Try different prompts and see how the answers change
- Add more data sources to the knowledge base
- Experiment with different LLM models (GPT-4o, Claude, Gemini, local models via Ollama)
- Try Elasticsearch as a search backend

# 1.11 Agents 

The LLM is in charge now, and it can:

- fix typos
- search again with different terms
- ask the user a clarifying question

In Part 2 of this module, we'll cover:

- Function calling, so we can give the LLM tools it can use
- The agentic loop, where the LLM decides when to call a tool, when to call another one, and when to stop and answer
- Frameworks, the libraries that run this loop for you

# 1.12 Quick RAG Revision (Optional) 

In [1]:
# wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
# wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

## Setting up RAG

In [2]:
import os 

from openai import OpenAI
# from dotenv import load_dotenv

# load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
from learning_code.rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

## Testing it

In [5]:
assistant.rag("How do I run Ollama locally?")

response.usage.input_tokens
806


'To run Ollama on your own machine you need to:\n\n1. **Install Ollama**  \n   - **macOS** – Download the\u202f`.pkg` file from\u202f<https://ollama.com/download> and run the installer.  \n   - **Windows** – Download the\u202f`.msi` file from the same page and install it.  \n   - **Linux** – Execute the install script in a terminal:  \n\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**  \n   Open a terminal (or command prompt) and run:\n\n   ```bash\n   ollama run llama3\n   ```\n\n   - This command downloads the LLaMA\u202f3 model (≈\u202f4\u202fGB) if it isn’t already present.  \n   - It then starts the model and gives you an interactive, chat‑like prompt where you can type questions.\n\n3. **Verify the local Ollama server**  \n\n   In another terminal (or a new tab) run:\n\n   ```bash\n   curl http://localhost:11434\n   ```\n\n   You should get a JSON response such as:\n\n   ```json\n   {"models": [...]}\n   ```\n\n   This co

In [6]:
assistant.rag("How do I run Olama locally?")

response.usage.input_tokens
949


'Yes—you can run Olama locally instead of using Codespaces.  \nTo do so you’ll need to set up the same tools that the course relies on, for example:\n\n* Python (the version required by the notebooks)  \n* `uv` for dependency management  \n* Jupyter (to run the notebooks)  \n* Docker (if any of the modules use containerised services)  \n* Any other utilities the specific module mentions\n\nIf you choose to run locally, be sure to **document your setup steps** and keep the environment reproducible (for example, by using a `uv` lock file or a `requirements.txt`). This makes it easy to rebuild the environment later or share it with others.'

# 1.13 Function Calling

The pipeline is fixed: search, build prompt, LLM.

In [7]:
# def rag(self, query):
#     search_results = self.search(query)
#     prompt = self.build_prompt(query, search_results)
#     answer = self.llm(prompt)
#     return answer

LLM is a passenger here, not a driver -> never sees the bad search results, so it can't try again with a corrected query.

## The agent alternative

The difference is about who makes the decisions:

- With RAG, the developer decides. We fix the steps up front, so search always runs once with the exact user query.
- With an agent, the LLM decides. It chooses which actions to take and when to stop.

## Asking without tools

In [36]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    input=messages
)

response.output_text

"I'd be happy to help! However, I need more information. Could you please provide more context about the course you're interested in joining? Here are a few questions to help me better understand:\n\n1. What is the name of the course?\n2. Is it an online or offline course?\n3. Is it a free or paid course?\n4. What are the prerequisites for joining the course (if any)?\n\nOnce I have more information, I can try to help you determine if you can still join the course."

## Defining the tool

In [9]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [10]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

## Sending the question with the tool

In [37]:
print(messages)

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]


In [38]:
response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseReasoningItem(id='resp_01kvqznb9wesvvppn2z97ybpvn', summary=[], type='reasoning', content=[Content(text='The user asks: "I just discovered the course. Can I join it?" We need to answer based on the policies and presumably provide info about enrollment. No personal data. So we can respond: yes, you can join, enrollment open, steps, prerequisites, etc.\n\nWe might need to check FAQ: maybe there\'s a FAQ entry about enrollment. Use search function.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"enrollment open can I join the course"}', call_id='fc_9a18ba17-07c9-4da4-bae2-c085f63b8689', name='search', type='function_call', id='fc_9a18ba17-07c9-4da4-bae2-c085f63b8689', namespace=None, status='completed')]

In [31]:
response.output_text

''

## Executing the function and sending the result back

In [39]:
import json

# with Groq it's 1 not 0 the call arguments 
call = response.output[1]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [33]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "aa310de435",
    "course": "llm-zoomcamp",
    "section": "Module 1: RAG"

In [40]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

## Asking the model again

In [41]:
response = openai_client.responses.create(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseReasoningItem(id='resp_01kvqznszaezgtzp1fns4f5kv7', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed'),
 ResponseOutputMessage(id='msg_01kvqznszaezh907x40z5cqw8g', content=[ResponseOutputText(annotations=[], text='Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', type='output_text', logprobs=None)], role='assistant', status='completed', type='message', phase=None)]

In [42]:
response.output_text

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

## Token usage and cost

In [19]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(1444, 25)

In [20]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(1332, 30)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0002178


# 1.14 The Agentic Loop

## Anatomy of an agent

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the `developer` message. The better the instructions, the better the agent helps.
- Tools, the functions the agent can call to carry out the task. For us that's only `search`.
- Memory, the message history. We append every prompt, every model output, and every tool result. The agent reads this to know what it has already tried.

## A developer prompt

In [43]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

## A function-call helper

In [44]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

## Processing one response

In [45]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"Can I join the course enrollment policy"}


## The full agent loop

In [46]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes, you can join the course. The course is self-paced, and you can start learning and submitting homework without registering. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions and complete the peer-review process. You can find more information on the course website and start with the course materials and GitHub repository.


## Wrapping it in a function

In [47]:
def agent_loop(instructions, question, model="openai/gpt-oss-120b") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [48]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama run locally"}
iteration #2...
function_call: search {"query":"run Ollama locally"}
iteration #3...
ASSISTANT:
**Running Ollama locally – step‑by‑step**

Below is a concise checklist that gets a local Ollama server up and running, then shows how you can talk to it from the command line or from Python code. The instructions work on macOS, Windows, and Linux.

---

## 1. Install the Ollama binary

| OS | Command / Action |
|----|------------------|
| **macOS** | Download the `*.pkg` from <https://ollama.com/download> and double‑click to install. |
| **Windows** | Download the `*.msi` from the same page and run the installer. |
| **Linux** | Run the one‑liner installer: <br>`curl -fsSL https://ollama.com/install.sh \| sh` |

*The installer puts an `ollama` executable on your `$PATH` (or adds a shortcut on Windows).*

---

## 2. Pull & run a model (e.g., Llama 3)

```bash
# This will download the model (≈4 GB) the first time and start th

'**Running\u202fOllama locally – step‑by‑step**\n\nBelow is a concise checklist that gets a local Ollama server up and running, then shows how you can talk to it from the command line or from Python code. The instructions work on macOS, Windows, and Linux.\n\n---\n\n## 1. Install the Ollama binary\n\n| OS | Command / Action |\n|----|------------------|\n| **macOS** | Download the `*.pkg` from <https://ollama.com/download> and double‑click to install. |\n| **Windows** | Download the `*.msi` from the same page and run the installer. |\n| **Linux** | Run the one‑liner installer: <br>`curl -fsSL https://ollama.com/install.sh \\| sh` |\n\n*The installer puts an `ollama` executable on your `$PATH` (or adds a shortcut on Windows).*\n\n---\n\n## 2. Pull & run a model (e.g., Llama\u202f3)\n\n```bash\n# This will download the model (≈4\u202fGB) the first time and start the server\nollama run llama3\n```\n\n- The command starts a **local HTTP server** on port **11434**.\n- You’ll see a small REPL

In [49]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"Can I still join the course after it started"}
iteration #2...
ASSISTANT:
**Yes – you can still join the LLM Zoomcamp even if you discovered it after the start date.**  

Here’s what you need to know:

| What you can do now | What you need to watch out for |
|----------------------|--------------------------------|
| **Start learning immediately** – all videos, notebooks, and the GitHub repository are publicly available, so you can begin working on the lessons and homework right away. | **Certificate eligibility** – to earn a certificate you must submit a completed capstone project *while the course is still accepting submissions*. The deadline for project submissions is the same for everyone in the live cohort. |
| **Register (optional)** – registration is just a way for the organizers to gauge interest; it isn’t required to access the material or submit homework. | **Peer‑review requirement** – after you submit your capstone you’ll be a

'**Yes – you can still join the LLM\u202fZoomcamp even if you discovered it after the start date.**  \n\nHere’s what you need to know:\n\n| What you can do now | What you need to watch out for |\n|----------------------|--------------------------------|\n| **Start learning immediately** – all videos, notebooks, and the GitHub repository are publicly available, so you can begin working on the lessons and homework right away. | **Certificate eligibility** – to earn a certificate you must submit a completed capstone project *while the course is still accepting submissions*. The deadline for project submissions is the same for everyone in the live cohort. |\n| **Register (optional)** – registration is just a way for the organizers to gauge interest; it isn’t required to access the material or submit homework. | **Peer‑review requirement** – after you submit your capstone you’ll be asked to peer‑review three other projects. This can only be done while the cohort is active. |\n| **Join the c

## Encouraging multiple searches

In [50]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"Can I join the course? enrollment eligibility prerequisites"}
iteration #2...
ASSISTANT:
**Yes – you can still join the LLM Zoomcamp!**  

- **Enrollment:** There’s no formal gate‑keeping. If you want to be part of a live cohort (so you can earn a certificate), just register through the [registration form](https://courses.datatalks.club/llm-zoomcamp-2026/). Even if you miss the form deadline you can still start learning and submit homework while the submission window is open.  

- **Certificate:** To receive a certificate you need to:
  1. Complete the full course in a live cohort (self‑paced mode does **not** qualify).  
  2. Submit your final project before the deadline.  
  3. Peer‑review three other participants’ projects (this only happens while the cohort is running).

- **Getting started right away:**  
  1. Browse the **LLM Zoomcamp documentation**: https://datatalks.club/docs/courses/llm-zoomcamp/  
  2. Check the **course logist

'**Yes – you can still join the LLM Zoomcamp!**  \n\n- **Enrollment:** There’s no formal gate‑keeping. If you want to be part of a live cohort (so you can earn a certificate), just register through the\u202f[registration form](https://courses.datatalks.club/llm-zoomcamp-2026/). Even if you miss the form deadline you can still start learning and submit homework while the submission window is open.  \n\n- **Certificate:** To receive a certificate you need to:\n  1. Complete the full course in a live cohort (self‑paced mode does **not** qualify).  \n  2. Submit your final project before the deadline.  \n  3. Peer‑review three other participants’ projects (this only happens while the cohort is running).\n\n- **Getting started right away:**  \n  1. Browse the **LLM Zoomcamp documentation**: https://datatalks.club/docs/courses/llm-zoomcamp/  \n  2. Check the **course logistics** page for weekly deadlines: https://datatalks.club/docs/courses/zoomcamp-logistics/  \n  3. Clone the **GitHub repo

## Restricting off-topic questions

In [51]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit definition chess opening"}
iteration #2...
function_call: search {"query":"Queen's Gambit opening chess what is it"}
iteration #3...
function_call: search {"query":"queen's gambit definition in chess opening"}
iteration #4...
ASSISTANT:
**Queen’s Gambit – a quick‑look overview**

| Aspect | What it is |
|--------|------------|
| **Opening moves** | 1 d4 d5 2 c4 |
| **Idea** | White offers the pawn on c4 to tempt Black’s d‑pawn away from the centre. If Black accepts, White can gain rapid development and strong central control. |
| **Why it’s called a “gambit”** | In a true gambit a piece is deliberately sacrificed for long‑term compensation. In the Queen’s Gambit the pawn is almost always recoverable, so many authors treat it more as a “positional” opening than a risky sacrifice. |
| **Main branches** | 1. **Queen’s Gambit Accepted (QGA)** – 2…dxc4. Black grabs the pawn; White usually regains it later while developing pieces (

'**Queen’s Gambit – a quick‑look overview**\n\n| Aspect | What it is |\n|--------|------------|\n| **Opening moves** | 1\u202fd4\u202fd5\u202f2\u202fc4 |\n| **Idea** | White offers the pawn on c4 to tempt Black’s d‑pawn away from the centre. If Black accepts, White can gain rapid development and strong central control. |\n| **Why it’s called a “gambit”** | In a true gambit a piece is deliberately sacrificed for long‑term compensation. In the Queen’s Gambit the pawn is almost always recoverable, so many authors treat it more as a “positional” opening than a risky sacrifice. |\n| **Main branches** | 1. **Queen’s Gambit Accepted (QGA)** – 2…dxc4. Black grabs the pawn; White usually regains it later while developing pieces (e.g.,\u202f3\u202fNc3\u202fNf6\u202f4\u202fe3\u202fe6\u202f5\u202fBxc4).<br>2. **Queen’s Gambit Declined (QGD)** – 2…e6 or 2…c6 (the “Slav Defense”). Black keeps the pawn structure solid and fights for the centre. <br>3. **Other popular declines** – 2…c5 (the “Albin Cou

In [52]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searches.

The question has to be about the course or its logistics, offtopic questions
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
I’m sorry, but I don’t have information on that topic in the course FAQ. If you have any questions about the course content, assignments, schedule, or other course‑related matters, feel free to ask!


'I’m sorry, but I don’t have information on that topic in the course FAQ.\u202fIf you have any questions about the course content, assignments, schedule, or other course‑related matters, feel free to ask!'

# 1.15 ToyAIKit

## Setup

In [53]:
!uv add toyaikit

Resolved 129 packages in 69ms
Audited 125 packages in 208ms


In [54]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

## Registering the tool

In [55]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

## Letting ToyAIKit generate the schema

In [56]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [58]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

## The chat interface and runner

In [60]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(client=openai_client, 
                            model="openai/gpt-oss-120b")
)

## Running one prompt

In [64]:
import warnings
warnings.filterwarnings("ignore", message=".*No pricing data.*")

In [65]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


## Cost and tokens

In [66]:
result.cost

## Continuing the conversation

In [67]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


Step,Command,What it does
Pick model,ollama list or browse Ollama site,Identify the exact model name
Pull (optional),ollama pull <model>,Download the model ahead of time
Run,ollama run <model>,Start the server with that model
Python call,"model=""<model>"" in ollama.chat",Use the same model in code


## Interactive chat

In [ ]:
# runs indefinitely
# runner.run()

# 1.16 Other Frameworks

## OpenAI Agents SDK

In [ ]:
!uv add openai-agents

## PydanticAI

In [ ]:
!uv add pydantic-ai

## LangChain / LangGraph

## Google ADK

## Others

Other frameworks worth knowing:

- CrewAI - multi-agent orchestration
- AutoGen - multi-agent conversations from Microsoft
- Semantic Kernel - from Microsoft, supports C# and Python
- Smolagents - lightweight agent framework from HuggingFace
- Anthropic Tool Use - Anthropic's native tool use API

Pick one that fits your stack and your needs. The hard part is designing good tools and prompts - the loop is always the same.

## Avoiding agents when you can

Adding an agent is a real cost:

- More API calls per request, since the loop can fire many tool calls before the model is satisfied.
- Higher latency, since each round-trip waits on the model.
- More money spent, since every iteration is another billed call and the full message history is re-sent each turn.
- More moving parts, since you now monitor cost, iteration count, and whether the agent is going in circles.
- Less predictable behavior, since the LLM decides what to do and two runs of the same prompt can take different paths.

Before reaching for an agent, ask whether the problem needs one. A lot of tasks are well served by simpler approaches.

Reach for one of these first:

- plain RAG, with one search and one answer
- parsing or templating a document into another form
- a single LLM call with no tools

Try the simpler approach first, and if it works, ship it. Reach for the agent loop only when you've tried the simpler solution and it genuinely can't handle the problem. By then you'll know the extra complexity is worth it, and you'll be ready to take it on.